In [26]:
from openai import OpenAI
import os
import re
import time

In [27]:
# Initialize the OpenAI client with API key.
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # specified value was saved

if client.api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable not set")

In [28]:
# Read the few-shot prompt from the external file
prompt_path = 'C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/data annotation/annotation_prompt.txt'

with open(prompt_path, 'r') as file:
    PROMPT = file.read()

In [29]:
def read_sentences(file_path):
    # Read the entire content of the file
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()

    # Split the content into sentences
    sentences = re.split(r'(?<=[.!?]) +', content)

    return sentences


def dynamic_chunking(sentences, start_index=1):
    """ 
    This function dynamically chunks a list of sentences into groups based on their content and structure. 
    Each sentence is checked if it ends with a question mark. If it does, the function attempts to include up to two subsequent sentences in the current chunk. 
    
    """
    chunks = []
    i = 0
    
    while i < len(sentences):
        current_chunk = []
        sentence = sentences[i]
        
        # Use start_index to adjust the sentence ID
        current_chunk.append((start_index + i, sentence))
        
        if sentence.strip().endswith('?'):
            if i + 2 < len(sentences):
                current_chunk.append((start_index + i + 1, sentences[i + 1]))
                current_chunk.append((start_index + i + 2, sentences[i + 2]))
                i += 2
            elif i + 1 < len(sentences):
                current_chunk.append((start_index + i + 1, sentences[i + 1]))
                i += 1

        chunks.append(current_chunk)
        i += 1

    return chunks


def annotate_with_spo_labels(PROMPT, text_tuples):
    """ 
    Annotates in a token-level each conversational sentence with SPO labels.
    
    """
    formatted_text = "\n".join(f"Sentence ID: {tid}\n{txt}" for tid, txt in text_tuples)
    response = client.chat.completions.create(
        model = "gpt-4o",
        messages = [
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": formatted_text}
        ],
        max_tokens = 4096,
        temperature = 0.2  # affects the randomness or creativity of the responses produced by the model
    )

    if response.choices:
        api_output = response.choices[0].message.content

        unwanted_prefix = "```json\n["
        unwanted_suffix = "```"
        if api_output.startswith(unwanted_prefix):
            api_output = api_output[len(unwanted_prefix):]  # Remove the specific prefix
        if api_output.endswith(unwanted_suffix):
            api_output = api_output[:-len(unwanted_suffix)]  # Remove the specific suffix
        return api_output.strip()  # Remove any extra whitespace
    else:
        raise ValueError("No valid response received from the API")


def save_annotations(annotations, output_file):
    try:
        with open(output_file, 'w', encoding='utf-8') as file:
            # Start the JSON array
            file.write("[\n")
            
            # Write annotations as continuous entries in the JSON array
            for i, annotation in enumerate(annotations):
                # Remove the surrounding square brackets from each annotation chunk if present
                if annotation.startswith('[') and annotation.endswith(']'):
                    annotation = annotation[1:-1]  # Remove the first '[' and last ']'
                
                # Add a comma between entries, but not before the first entry
                if i > 0:
                    file.write(",\n")
                
                file.write(annotation)
            
            # Close the JSON array
            file.write("\n]")
            
        print("Annotations saved successfully.")
    except Exception as e:
        print(f"Failed to save annotations: {e}")


def main(file_path, prompt, output_file, start_index=810):
    try:      
        all_sentences = read_sentences(file_path)
        chunks = dynamic_chunking(all_sentences, start_index=start_index)

        annotations = []
        for chunk in chunks:
            response = annotate_with_spo_labels(prompt, chunk)
            annotations.append(response)
            time.sleep(1)
            save_annotations(annotations, output_file)
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")


In [30]:
main('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/data preprocessing/data/preprocessed_train_data.txt', PROMPT, 'C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/data annotation/annotated data/annotated_train_data.json')

Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotations saved successfully.
Annotati